<a href="https://colab.research.google.com/github/Killian091/Machine_Learning/blob/main/Car_Price_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CAR** **PRICE** **PREDICTOR**

In [59]:
import pandas as pd
car = pd.read_csv("quikr_car.csv")

In [60]:
car.head()

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007,"80,000","45,000 kms",Petrol
1,Mahindra Jeep CL550 MDI,Mahindra,2006,"4,25,000",40 kms,Diesel
2,Maruti Suzuki Alto 800 Vxi,Maruti,2018,Ask For Price,"22,000 kms",Petrol
3,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014,"3,25,000","28,000 kms",Petrol
4,Ford EcoSport Titanium 1.5L TDCi,Ford,2014,"5,75,000","36,000 kms",Diesel


In [61]:
car.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 892 entries, 0 to 891
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        892 non-null    object
 1   company     892 non-null    object
 2   year        892 non-null    object
 3   Price       892 non-null    object
 4   kms_driven  840 non-null    object
 5   fuel_type   837 non-null    object
dtypes: object(6)
memory usage: 41.9+ KB


In [62]:
backup = car.copy()

In [63]:
car = car[car['year'].str.isnumeric()]

In [64]:
car['year'] = car['year'].astype(int)

In [65]:
car.info()

<class 'pandas.core.frame.DataFrame'>
Index: 842 entries, 0 to 891
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        842 non-null    object
 1   company     842 non-null    object
 2   year        842 non-null    int64 
 3   Price       842 non-null    object
 4   kms_driven  840 non-null    object
 5   fuel_type   837 non-null    object
dtypes: int64(1), object(5)
memory usage: 46.0+ KB


In [66]:
car = car[car['Price']!="Ask For Price"]

In [67]:
car['Price'] = car['Price'].str.replace(',','').astype(int)

In [68]:
car['kms_driven'] = car['kms_driven'].str.split(' ').str.get(0)

In [69]:
car= car[car['kms_driven']!="Petrol"]

In [70]:
car['kms_driven'] = car['kms_driven'].str.replace(',','').astype(int)

In [71]:
car = car[~car['fuel_type'].isna()]

In [72]:
car['name'] = car['name'].str.split(' ').str.slice(0,3).str.join(' ')

In [73]:
car.describe()

,year,Price,kms_driven
count,816.000000,8.160000e+02,816.000000
mean,2012.444853,4.117176e+05,46275.531863
std,4.002992,4.751844e+05,34297.428044
min,1995.000000,3.000000e+04,0.000000
25%,2010.000000,1.750000e+05,27000.000000
50%,2013.000000,2.999990e+05,41000.000000
75%,2015.000000,4.912500e+05,56818.500000
max,2019.000000,8.500003e+06,400000.000000


In [74]:
car = car[car['Price']<6e6]

In [75]:
car.info()

<class 'pandas.core.frame.DataFrame'>
Index: 815 entries, 0 to 889
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        815 non-null    object
 1   company     815 non-null    object
 2   year        815 non-null    int64 
 3   Price       815 non-null    int64 
 4   kms_driven  815 non-null    int64 
 5   fuel_type   815 non-null    object
dtypes: int64(3), object(3)
memory usage: 44.6+ KB


In [76]:
car = car.reset_index(drop=True)

In [77]:
car.to_csv('Cleaned Car.csv')

## **Model** **Training**

In [78]:
X = car.drop(columns='Price')
Y = car['Price']

## **Train** **Test** **Split**

In [79]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(X,Y, test_size=0.2)

In [80]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [81]:
ohe = OneHotEncoder()
ohe.fit(X[['name','company','fuel_type']])

OneHotEncoder()

In [82]:
column_trans = ColumnTransformer(
    [('ohe', ohe, ['name', 'company', 'fuel_type'])],
    remainder='passthrough')

In [83]:
lr= LinearRegression()

In [84]:
pipe = Pipeline([('preprocessor', column_trans),('regressor',lr)])

In [85]:
pipe.fit(X_train,Y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe', OneHotEncoder(),
                                                  ['name', 'company',
                                                   'fuel_type'])])),
                ('regressor', LinearRegression())])